In [ ]:
# Imports
import json
import numpy as np
import holoviews as hv
import panel as pn
import pandas as pd
import geopandas as gpd
import geoviews as gv
from geoviews import feature as gf
from shapely.geometry import Polygon
import cartopy.crs as crs  # Has to be imported after geoviews. Why? God only knows.

import multiprocessing as mp
from io import StringIO
from dask.diagnostics.progress import ProgressBar, format_time, default_timer
import inspect
import xarray as xr
import xclim as xc
from xclim.core.indicator import Indicator
from xclim import subset as xcsub
from xclim import ensembles as xcens
import threddsclient as tds
import hvplot.xarray
import tempfile
from pathlib import Path
from time import sleep

gv.extension('bokeh')
#hv.extension('bokeh')
pn.extension()

# Portraits climatiques intéractifs

This draft implementation divides the workload in three panels:

- Region extraction
- Indicator computation
- Plotting

In order to have the three panels relatively independent and to have as few global variables as possible, I tried to have most of the info and metadata written to files. The structure should be as follows:


    nc_data : The main data folder
        regions : Folder containing region maps
            xyz.json : GeoJSON defining polygons for user selection in the first panel
                       also has metadata for i18n
            xyz_simp.json : [Optional] GeoJSON defining the same polygons but 
                            in a lower definition to make loading faster
        abc_X : Extracted region folder
            regions_bounds.geojson : Definition of the region polygon and name
            sub_MODEL_rcp??.nc : Extracted data for one model and one rcp
            ind_INDICATOR_obs/sim.nc : Indicator computed over the region.
                                       "sim" is the ensemble statistics.          

There are three ways for the user to select a region for extraction:

- Drawing : Using bokeh's polygon drawing tool. The data folder will be named `userdrawn_N`, where N is the first free integer.
- Importing : Using a user-provided fiona-readable file. Multiple polygons are merged as one "Multipolygon". The data folder will be named `userimported_N`.
- Selecting : The user can select a meta-map of regions (those defined in the `regions` folder) and select one of more polygons from it. The folder will be named `xyz_N`, where `xyz` is the meta-map file name and N is the index of the region in the map. Multiple region selection will give : `xyz_N_M_O_P`.

In [17]:
# Initial state and constants
state = {'region': None}  # Only used by the region extractor
crs_merc_pp = crs.Mercator().proj4_params
data_folder = Path('nc_data')
# Copied from my scenario_web scripts
# Assumes YS, AS=JUL or QS-DEC, translates the month to a season name
season_label = {1: 'Annuel', 3: 'Printemps', 6: 'Ete', 7: 'Annuel', 9: 'Automne', 12: 'Hiver'}

def read_region_map(filename):
    """Read a region map and metadata.
    
    Returns a dict with:
     
     - name_fr : A human-readable name in french
     - name_en : A human-readable name in english
     - identifier : The identifier use in folder naming, should be the same as the filename
     - hidef : The geojson file name
     - lowdef : A geojson file name defining lower resolution versions of the polygons

    The first three fields should be defined at the top-level of the hidef geojson file.
    """
    regions = {}
    regions['hidef'] = filename
    with filename.open('r', encoding='utf-8') as f:
        # [Geo]pandas doesn't have a metadata entry...
        d = json.load(f)
    regions.update(name_fr=d['name_fr'], name_en=d['name_en'], identifier=d['identifier'])
    if (filename.parent / (filename.stem + '_simp.json')).is_file():
        regions['lowdef'] = filename.parent / (filename.stem + '_simp.json')
    else:
        regions['lowdef'] = regions['hidef']
    return regions


def load_extracted(folder):
    """Load metadata from an extracted data folder. 
    For now doesn't do much...
    """
    region = gpd.read_file(folder / 'region_bounds.geojson')
    return region

# Load all meta regions
predef_regions = [read_region_map(file)
                  for file in Path('nc_data/regions/').iterdir()
                  if file.suffix == '.json' and '_simp' not in file.stem]

In [ ]:
# Task/Dask callback and progress.
class Progress(ProgressBar):
    """Object to control a progress panel row with a title, progress bar and elapsed time"""
    def __init__(self, bar):
        """bar is a panel.Row with Markdown, panel.widgets.Progress and Markdown"""
        self.pb = bar[1]
        self.bar = bar
        super().__init__(out=StringIO(), dt=2)
    
    def _finish(self, dsk, state, errored):
        self.finish(errored)
        super()._finish(dsk, state, errored)
    
    def _draw_bar(self, frac, elapsed):
        """Change the bar's value and the elapsed time string"""
        self.pb.value = int(frac * 100)
        self.bar[2] = format_time(elapsed)
    
    def new_task(self, name):
        """Not needed for dask tasks, initialize the progress bar with a new title."""
        self.pb.bar_color = 'primary'
        self.pb.value = None
        self.pb.active = True
        self.bar[2]  = ""
        self.bar[0] = f'*{name}*'
        self._start_time = default_timer()

    def update(self, frac):
        """When not using dask tasks, update the bar to a given frac (0..1)"""
        elapsed = default_timer() - self._start_time
        self._draw_bar(frac, elapsed)
    
    def finish(self, errored=False):
        """Useless with dask's task, mark the task as finished."""
        self.update(1)
        if errored:
            self.pb.bar_color = 'danger'
            self.bar[0] = f'*Last task exited with errors*'
        else:
            self.pb.bar_colors = 'success'
        self.pb.active = False

# A row with a progress bar widget
w_progbar = pn.widgets.Progress(name='Tasks Progress', active=False, width=300, bar_color='primary')
pbar = Progress(pn.Row('*No tasks to compute.*', w_progbar, ''))
# pbar.register() #<-- This makes Firefox slow down a lot. Any use with dask does that...

In [13]:
# Data management
data_config = {
    # Normal path : 'https://pavics.ouranos.ca/twitcher/ows/proxy/thredds/catalog/datasets/simulations/bias_adjusted/cmip5/ouranos/cb-oura-1.0/catalog.html',
    # This path by-passes twitcher and makes things faster. Also this "rechunk" has fewer datasets, but reads faster
    'baseurl_scenarios': 'http://pavics.ouranos.ca:8083/twitcher/ows/proxy/thredds/catalog/birdhouse/testdata/ouranos/cb-oura-1.0_rechunk/catalog.html',
    'url_obs': 'https://pavics.ouranos.ca/twitcher/ows/proxy/thredds/dodsC/datasets/gridded_obs/nrcan.ncml',
    'datasets': None  # List of OpenDap urls, populated when needed
}


def get_folder_name(name):
    """Return the proper folder name, adding a suffix to make the name unique if needed."""
    if name is None:
        n = len(list(Path('nc_data').glob('tmp*')))
        return Path(tempfile.mktemp(prefix='tmp', dir='nc_data')).stem, n
    else:
        if name.startswith('user'):
            n = len(list(Path('nc_data').glob(name + '*')))
            return Path(tempfile.mktemp(prefix=name, dir='nc_data')).stem, n
        else:
            return name, None

def _parse_data_url(url):
    """Extract model and rcp info from Opendap url"""
    if 'rechunk' in url:
        parts = url.split('/')
        model = parts[-4]
        rcp = parts[-3]
    else:
        parts = url.split('/')[-1].split('_')
        model = parts[1]
        rcp = parts[2].split('+')[-1]
    return model, rcp, url

def get_all_datasets():
    """Returns a list of all sim opendap urls, populating the config if needed"""
    if data_config['datasets'] is None:
        pbar.new_task('Crawling the server to find all datasets.')
        datasets = []
        i = 0
        for ds in tds.crawl(data_config['baseurl_scenarios'], depth=10):
            if '.ncml' in ds.name:
                datasets.append(_parse_data_url(ds.opendap_url()))
                i += 1
                pbar.update(i / 23)

        datasets.append(('NRCAN', 'obs', data_config['url_obs']))
        pbar.finish()
        data_config['datasets'] = datasets
    return data_config['datasets']

def _extract(model, rcp, url, folder, region):
    """Does the extraction of a region from one model / rcp."""
    filepath = folder / f'sub_{model}_{rcp}.nc'
    if filepath.is_file():
        return filepath
    ds = xr.open_dataset(url, drop_variables=['ts', 'time_vectors'],
                         # These chunks are optimal (?) for the "rechunk" urls
                         chunks={'time': 256,
                                 'lat': 16, 'lon': 16})
    # Mask creation could be made once? here it does the same for all models...
    # Only the obs might be different
    mask = xcsub.create_mask(x_dim=ds.lon, y_dim=ds.lat, poly=region)
    ds.where(mask.notnull(), drop=True).to_netcdf(filepath, compute=True)
    return filepath

def extract_region(region):
    """Extract a region from all available datasets
    Region is a geopandas dataframe with one element
    """
    datasets = get_all_datasets()
    # The "id" field should already have an unique name
    folder = Path('nc_data') / region.id[0]
    if not folder.is_dir():
        folder.mkdir()
    
    # Extraction doesn't max neither cpu usage nor internet speed
    # So it is faster to launch a few in parallel
    manager = mp.Manager()
    queue = manager.Queue()
    pool = mp.Pool(4)
    pbar.new_task('Extracting region.')

    jobs = []
    for model, rcp, url in datasets:
        job = pool.apply_async(_extract, (model, rcp, url, folder, region))
        jobs.append(job)
        # The first seconds of the job do max the cpu, so waiting relaxes the machine stress a bit
        # evidently, not an optimal way to do this
        sleep(2)
    
    for i, job in enumerate(jobs):
        filepth = job.get()
        pbar.update((i + 1)  / len(jobs))
        
    # The existence of this file signifies that the folder is ready for computation
    # We wait for all other extraction to be successful before writing it
    region.to_file(folder / 'region_bounds.geojson', driver='GeoJSON')
    pbar.finish()

def compute_indicator(ind, params):
    """Compute an indicator on all available regions
    
    ind is an xc.core.indicator.Indicator instance
    params is a dict of kwargs
    """
    invars = []
    # Not optimal, assumes all non-optional arguments are data variables
    for name, param in ind._sig.parameters.items():
        if param.default is inspect._empty:
            invars.append(name)

    for reg in list_available().keys():
        pbar.new_task(f'Computing indicator {ind.title} on region {reg}')
        obs = xr.open_dataset(Path('nc_data') / reg / 'sub_NRCAN_obs.nc', chunks={})
        rcp45 = xcens.create_ensemble(list((Path('nc_data') / reg).glob('sub_*_rcp45.nc')), chunks={}).chunk({'realization': -1})
        rcp85 = xcens.create_ensemble(list((Path('nc_data') / reg).glob('sub_*_rcp85.nc')), chunks={}).chunk({'realization': -1})
    
        if 'tas' in invars:
            obs = obs.assign(tas=xc.atmos.tg(tasmin=obs['tasmin'], tasmax=obs['tasmax']))
            rcp45 = rcp45.assign(tas=xc.atmos.tg(tasmin=rcp45['tasmin'], tasmax=rcp45['tasmax']))
            rcp85 = rcp85.assign(tas=xc.atmos.tg(tasmin=rcp85['tasmin'], tasmax=rcp85['tasmax']))

        dobs = unstack_yr_season(ind(*[obs[vr] for vr in invars], **params))
        drcp45 = unstack_yr_season(ind(*[rcp45[vr] for vr in invars], **params))
        drcp85 = unstack_yr_season(ind(*[rcp85[vr] for vr in invars], **params))
        
        # Merge all sims together and compute percentiles
        dsim = xr.concat((drcp45, drcp85), xr.DataArray(['rcp45', 'rcp85'], dims=('rcp',), name='rcp'))
        dsim.name = ind.var_name
        dsim_perc = xcens.ensemble_percentiles(dsim.to_dataset())
        
        dobs.to_netcdf(Path('nc_data') / reg / f'ind_{dobs.name}_obs.nc')
        dsim_perc.to_netcdf(Path('nc_data') / reg / f'ind_{dsim.name}_sim.nc')
        
def unstack_yr_season(ds):
    """Translate resampled data to a multi-index year [int] / season [string]"""
    ds['time'] = pd.MultiIndex.from_arrays([ds.time.dt.year.values + (ds.time.dt.month.values == 12).astype(int),
                                            [season_label[m] for m in ds.time.dt.month.values]],
                                           names=['year', 'season'])
    return ds.unstack('time').rename(year='time')
    
def list_available():
    """List available regions and indicators
    
    Returns dict with:
        region_name: (folder name)
            name : humand-readable region name
            indicator_filename : indicator's long name
    """
    avail = {}
    for folder in Path('nc_data').iterdir():
        if folder.name == 'regions' or not folder.is_dir():
            continue
        if ((folder / 'region_bounds.geojson').is_file()
            and len(list(folder.glob('sub_*.nc'))) > 5):
            regs = gpd.read_file(folder / 'region_bounds.geojson').iloc[0]
            avail[folder.name] = {'name': regs['name']}
        for indfile in folder.glob('ind_*_sim.nc'):
            if (folder / indfile.name.replace('sim', 'obs')).is_file():
                ind = xr.open_dataset(indfile, chunks={})
                avail[folder.name][indfile.name] = list(ind.data_vars.values())[0].attrs['long_name']
    return avail


# Panel making and callbacks

In [5]:
# Panel 1 : Region selection and data extraction.
# For all panel making I tried to follow this syntax in the variable names:
# w_* : panel widget
# C_* : panel Column
# R_* : panel row
# P_* : Panel object
# M_* : Map (holoviews plots
# handle_* : Function executed after a widget change

reg_list = {'-- Select an option -- ' : 'none'}
reg_list.update({r['name_en']: [r] for r in predef_regions})
reg_list.update({'-- Import your own geojson file -- ': 'import', 
                 '-- Draw your own region --' : 'polydraw'})

w_regsel = pn.widgets.Select(name='Region',
                             options=reg_list)

w_dataexbutt = pn.widgets.Button(name='Extract data', button_type='warning')
w_regname = pn.widgets.TextInput(name='Region name', value='')
C_dataextract = pn.Column(w_regname, w_dataexbutt)

w_addregbutt = pn.widgets.Button(name='Add region', button_type='primary')
w_cancelbutt = pn.widgets.Button(name='Cancel', button_type='danger')

# Extend of the Scénarios Génériques data
p = Polygon([(-89.04521, 40.04104), (-89.04521, 66.62331), (-54.46326, 66.62331), (-54.46326, 40.04104)])
gbase = gpd.GeoDataFrame({'id': [0], 'name': ['Scenario Generique v1 bounds']}, geometry=[p], crs='epsg:4326')
M_lims = gv.Polygons(gbase).opts(gv.opts.Polygons(fill_alpha=0))
base_proj = crs.Sinusoidal(central_longitude=-70)
lims = base_proj.transform_points(crs.PlateCarree(), np.array([-95, -30]), np.array([38, 68]))
M_base = (gf.land * gf.lakes * gf.ocean * gf.borders * gf.rivers)
M_tiles = gv.tile_sources.CartoEco

w_reg_avail = pn.widgets.MultiSelect(name='Available regions', value=[],
                                     options={dic.get('name', key): key for key, dic in list_available().items()},
                                     size=8)
## (Add / title), choice , (SubmitButton | fileInput | Nothing | name / data extr | title2), (cancel | list of available regions)
C_regopts = pn.Column(w_addregbutt, '', '## Already extracted regions', w_reg_avail)
P_reg = pn.Column(pbar.bar, pn.Row(
    C_regopts,
    (M_lims * M_base).opts(
        projection=base_proj, xlim=tuple(lims[:, 0]), ylim=tuple(lims[:, 1]), width=500, height=500 
    )
))

C_loading = pn.pane.GIF('https://upload.wikimedia.org/wikipedia/commons/b/b1/Loading_icon.gif')


def update_map(*regs, title=''):
    """Change the main panel map to show the polygons in input
    regs are strings (folder names) or Geopandas dataframes
    """
    polys = []
    for reg in regs:
        if isinstance(reg, str):
            polys.append(gpd.read_file(Path('nc_data') / reg / 'region_bounds.geojson'))
        else:
            polys.append(reg)
    poly = pd.concat(polys)
    bnds = poly.to_crs(crs_merc_pp).total_bounds
    map_bounds = dict(xlim=(bnds[0], bnds[2]), ylim=(bnds[1], bnds[3]))
    C_map = pn.Column(
        title, # '### You selected this region',
        (M_tiles * gv.Polygons(poly, vname='name').opts(
            alpha=0.25, fill_color=None, line_width=2, width=500, height=500
        ))
    )
    P_reg[1][1] = C_map

def handle_addreg(event):
    """When the Add Region button is clicked"""
    P_reg[1][1] = (M_lims * M_base).opts(
        projection=base_proj, xlim=tuple(lims[:, 0]), ylim=tuple(lims[:, 1]), width=500, height=500 
    )
    C_regopts[0] = '## Select a region'
    C_regopts[1] = w_regsel
    C_regopts[2] = ''
    C_regopts[3] = w_cancelbutt

w_addregbutt.on_click(handle_addreg)

def handle_cancel(event):
    """When the Cancel button is clicked"""
    P_reg[1][1] = (M_lims * M_base).opts(
        projection=base_proj, xlim=tuple(lims[:, 0]), ylim=tuple(lims[:, 1]), width=500, height=500 
    )
    C_regopts[0] = w_addregbutt
    C_regopts[1] = ''
    C_regopts[2] = '## Already extracted regions'
    C_regopts[3] = w_reg_avail
    state['region'] = None

w_cancelbutt.on_click(handle_cancel)

def handle_regchange(event):
    """When the Submit button is clicked"""
    P_reg[1][1] = C_loading
    def on_new_poly(reg):
        state['region'] = reg.to_crs(epsg=4326)
        # We should have a criteria for regions too big or too small
        area = reg.to_crs({'proj': 'cea'}).area[0] / 1e6
        update_map(reg, title='### You selected this region')
        w_regname.value = reg.id[0]
        C_regopts[2] = C_dataextract

    if event.obj.value == 'polydraw':
        
        poly = hv.Polygons([])
        poly_stream = hv.streams.PolyDraw(source=poly, drag=True, num_objects=1,
                                          show_vertices=True, styles={
                                          'fill_color': ['green']
                                          })

        M_regsel = (M_base * M_lims) * poly.opts(
            hv.opts.Polygons(fill_alpha=0.3, active_tools=['poly_draw'])
        ).opts(width=500, height=500)
        C_regsel = pn.Column('## Select a region on the map.\n'
                             'Double-click to start/stop drawing, click to add points.'
                             'The black square shows the base data\'s limits',
                             M_regsel)
        
        def on_submit_poly(event):
            raw_poly = [Polygon([(x, y) for x, y in zip(xs, ys)])
                        for xs, ys in zip(poly_stream.data['xs'], poly_stream.data['ys'])]
            poly = gpd.GeoDataFrame(geometry=raw_poly,
                                    crs=crs_merc_pp)
            folder_name, N = get_folder_name('userdrawn')
            poly = gpd.GeoDataFrame(
                {'id': [folder_name], 'name': [f'User drawn (#{N})']},
                geometry=[poly.unary_union],
                crs=poly.crs
            )
            
            on_new_poly(poly)

        w_regsubmit = pn.widgets.Button(name='Submit region', button_type='primary')
        w_regsubmit.on_click(on_submit_poly)
        C_regopts[2] = w_regsubmit
        P_reg[1][1] = C_regsel

    elif event.obj.value == 'import':
        def on_submit_file(event):
            event.obj.save('user_region.dat')
            poly = gpd.read_file('user_region.dat')
            folder_name, N = get_folder_name('userimported')
            poly = gpd.GeoDataFrame(
                {'id': [folder_name], 'name': [f'Imported region #{N}']},
                geometry=[poly.unary_union],
                crs=poly.crs
            )
            on_new_poly(poly)

        w_file = pn.widgets.FileInput(
            name='Choose a file containing one region (any fiona-supported files).'
        )
        w_file.param.watch(on_submit_file, 'value')
        C_regopts[2] = w_file
        P_reg[1][1] = M_lims * M_base 
    elif event.obj.value == 'none':
        pass
    else:
        # Region selection on the meta-map
        regs = event.obj.value[0]
        df_low = gpd.read_file(regs['lowdef'])
        polys = gv.Polygons(df_low, vdims=['name'])
        sel = hv.streams.Selection1D(source=polys)
        M_regsel = M_base * polys.opts(
            tools=['hover', 'tap'], color_index=None, color='skyblue',
            selection_color='steelblue', selection_line_color='k',
            nonselection_color='skyblue', nonselection_alpha=1, nonselection_line_color='k',
            hover_color='deepskyblue', hover_line_color='k'
        ).opts(width=500, height=500)
        C_regsel = pn.Column('### Select a region on the map.\n',
                             'Click to select, hold shift to select multiple regions.',
                             M_regsel)

        def on_submit_selection(event):
            P_reg[1][1] = C_loading
            df_high = gpd.read_file(regs['hidef'])
            poly = gpd.GeoDataFrame(
                {'id': [regs['identifier'] + '_' + '_'.join(map(str, sel.index))],
                 'name': [','.join(df_high.iloc[sel.index].name)]},
                geometry=[df_high.iloc[sel.index].unary_union],
                crs=df_high.crs
            )
            on_new_poly(poly)
        
        w_regsubmit = pn.widgets.Button(name='Submit selection', button_type='primary')
        w_regsubmit.on_click(on_submit_selection)
        C_regopts[2] = w_regsubmit
        P_reg[1][1] = C_regsel

w_regsel.param.watch(handle_regchange, 'value')


def handle_availsel(event):
    update_map(*event.obj.value, title='### Showing selected extracted regions')
    
w_reg_avail.param.watch(handle_availsel, 'value')

def handle_extract_data(event):
    # Set name of region
    state['region']['name'] = w_regname.value if w_regname.value else state['region']['id'][0]
    extract_region(state['region'])
    w_reg_avail.options = list(list_available().keys())
    handle_cancel(None)

w_dataexbutt.on_click(handle_extract_data)

In [18]:
# Panel 2: Indicator computation

# Indicator options
# TODO : Make use of xc.core.indicator.registry in xclim 0.19
# TODO:  Filter out the indicators that this notebook cannot compute (for example anything with other variables than tas, tasmin, tasmax and pr)
ind_list = {name : ind
            for name, ind in xc.atmos.__dict__.items()
            if isinstance(ind, Indicator)}
w_ind_sel = pn.widgets.Select(name='Indicator',
                              options=ind_list)

w_ind_compute = pn.widgets.Button(name='Compute indicator', button_type='warning')

w_ind_cancel = pn.widgets.Button(name='Cancel', button_type='danger')

w_ind_add = pn.widgets.Button(name='Add indicator', button_type='primary')

## (add | title), dropdown, indicator desc, opts column, ComputeButton, cancel
C_indopts = pn.Column(w_ind_add, '', '', '', '', '')

L_indicator = pn.Column('', '')

# List of all indicators and regions
C_list = pn.Column(pn.pane.Markdown('## Available regions and indicators.'), '')

def update_ind_avail():
    C_list[1] = '\n'.join([f' - {reg} : {ind}' for reg, inds in list_available().items() for key, ind in inds.items() if key != 'name'])
update_ind_avail()

def handle_ind_add(event):
    C_indopts[0] = '## Select an indicator'
    C_indopts[1] = w_ind_sel
    C_indopts[5] = w_ind_cancel

w_ind_add.on_click(handle_ind_add)

def handle_ind_cancel(event):
    C_indopts[0] = w_ind_add
    C_indopts[1] = ''
    C_indopts[2] = ''
    C_indopts[3] = ''
    C_indopts[4] = ''
    C_indopts[5] = ''

w_ind_cancel.on_click(handle_ind_cancel)

def handle_indicator_change(event):
    ind = event.obj.value
    
    opts = []
    for param in ind._sig.parameters.keys():
        defval = ind._sig.parameters[param].default
        if defval is inspect._empty:
            continue
        if param == 'freq':
            opts.append(pn.widgets.Select(name=param,
                                          options={'Annual': 'YS',
                                                   'Seasonal': 'QS-DEC'}))
        elif param == 'window':
            # intslider
            opts.append(pn.widgets.IntSlider(name=param, start=1, step=1, end=8, value=defval))
        else:
            opts.append(pn.widgets.TextInput(name=param,
                                             placeholder=str(defval)))
    C_indopts[2] = f'#### {ind.title}\n{ind.abstract}'
    C_indopts[3] = pn.Column(*opts)
    C_indopts[4] = w_ind_compute

w_ind_sel.param.watch(handle_indicator_change, 'value')

def handle_compute(event):
    # We should provide more feedback to the user about what is going on
    indicator = w_ind_sel.value
    
    params = {}
    for opt in C_indopts[3]:
        value = opt.value
        if not value:
            value = opt.placeholder
        params[opt.name] = value
    
    compute_indicator(indicator, params)
    update_ind_avail()
    handle_ind_cancel(None)

w_ind_compute.on_click(handle_compute)
P_ind = pn.Row(C_indopts, C_list)

In [7]:
# Panel 3 : Plotting
# TODO: Most. This panel is for now super simplistic, the goal would be to imitate and improve the graphs of the js website.
w_data_updlist_but = pn.widgets.Button(name='Update list', button_type='primary')
w_data_sel = pn.widgets.Select(options={'Empty list for now': None})
R_header = pn.Row('#### Current dataset : ', w_data_sel, w_data_updlist_but)


def update_data_avail(event=None):
    avail = {'-- Select a dataset --' : None}
    avail.update({f"{reg}: {ind}": data_folder / reg / indfile for reg, inds in list_available().items() for indfile, ind in inds.items()})
    w_data_sel.options = avail

update_data_avail()
w_data_updlist_but.on_click(update_data_avail)

T_data = pn.Tabs(('Raw maps', 'Please select data first'),
                 ('Horizon maps', 'Please select data first'),
                 ('Regional Timeseries', 'Please select data first'),
                 ('Summary statistics', "Ça va v'nir ça va v'nir là décourageons nous pas"),
                 width_policy='max')

def handle_data_change(event):
    if event.obj.value is None:
        return
    sim_file = event.obj.value
    ds_sim = xr.open_dataset(sim_file, decode_cf=False)
    ds_sim['time'] = xr.decode_cf(ds_sim).time
    # Workaround for percentiles:
    sim = xr.concat(ds_sim.data_vars.values(), xr.DataArray([10, 50, 90], dims=('percentiles',), name='percentiles'))
    
    obs_file = sim_file.parent / sim_file.name.replace('sim', 'obs')
    ds_obs = xr.open_dataset(obs_file, decode_cf=False)
    ds_obs['time'] = xr.decode_cf(ds_obs).time
    obs = list(ds_obs.data_vars.values())[0]
    
    state['current_data'] = {'sim': sim, 'obs': obs}

    # Raw maps
    T_data[0] = pn.Column(sim.hvplot.quadmesh('lat', 'lon', title='Raw sim'),
                           obs.hvplot.quadmesh('lat', 'lon', title='Raw obs'))

    # Horizon maps
    def group_horizons(da):
        return xr.concat((da.sel(time=slice('1981', '2010')).mean('time'),
                          da.sel(time=slice('2041', '2070')).mean('time'),
                          da.sel(time=slice('2071', '2100')).mean('time')),
                         xr.DataArray(['1981-2010', '2041-2070', '2071-2100'], dims=('horizon',), name='horizon'))
    sim_hor = group_horizons(sim)
    obs_hor = obs.sel(time=slice('1981', '2010')).mean('time')
    T_data[1] = pn.Column(sim_hor.hvplot.quadmesh('lat', 'lon', title='Simulations'),
                           obs_hor.hvplot.quadmesh('lat', 'lon', title='Observations'))
    
    # Regional timeseries
    sim_ts = sim.mean(['lat', 'lon'])
    obs_ts = obs.mean(['lat', 'lon'])
    reg_ts = xr.Dataset(data_vars={'sim': sim_ts, 'obs': obs_ts})
    T_data[2] = reg_ts.hvplot('time')

w_data_sel.param.watch(handle_data_change, 'value')

P_data = pn.Column(R_header, T_data, width_policy='max')

In [8]:
# All options:
P_reg

Column
    [0] Row
        [0] Markdown(str)
        [1] Progress(active=False, bar_color='primary', name='Tasks Progress', width=300)
        [2] Markdown(str)
    [1] Row
        [0] Column
            [0] Button(button_type='primary', name='Add region')
            [1] Markdown(str)
            [2] Markdown(str)
            [3] MultiSelect(name='Available regions', options={'MRC Bas-Saint-Laurent': ...}, size=8)
        [1] HoloViews(Overlay)

In [15]:
P_ind

Row
    [0] Column
        [0] Button(button_type='primary', name='Add indicator')
        [1] Markdown(str)
        [2] Markdown(str)
        [3] Markdown(str)
        [4] Markdown(str)
        [5] Markdown(str)
    [1] Column
        [0] Markdown(str)
        [1] Markdown(str)

In [10]:
P_data

Column(width_policy='max')
    [0] Row
        [0] Markdown(str)
        [1] Select(options={'-- Select a dataset --':...})
        [2] Button(button_type='primary', name='Update list')
    [1] Tabs(width_policy='max')
        [0] Markdown(str, name='Raw maps')
        [1] Markdown(str, name='Horizon maps')
        [2] Markdown(str, name='Regional Timeseries')
        [3] Markdown(str, name='Summary statistics')